# Phase3. Single Factor Testing

## 3.1 Loading Data & Packages

In [143]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import re
from pathlib import Path

CACHE_DIR = Path("data/cache/processed_fast")


In [24]:
def load_data_cache(cache_dir: Path = CACHE_DIR, names: list[str] | None = None):
    wanted = set(names) if names else None
    out = {}

    for fp in sorted(cache_dir.glob("*.parquet")):
        if wanted is not None and fp.stem not in wanted:
            continue
        out[fp.stem] = pd.read_parquet(fp, engine="pyarrow")

    return out

In [25]:
aligned_data = load_data_cache(
    names=[
        "cap", "close", "open", "high", "low", "volume", "vwap", "returns",
        "shares_outstanding", "debt", "operating_income", "pe", "pb", "net_profit"
    ]
)

In [53]:
industry = pd.read_parquet('/Users/apple/Desktop/PitchBook/Multi-Factor L:S/Processed_Data/industries.parquet').iloc[:, 0]
subindustry = pd.read_parquet('/Users/apple/Desktop/PitchBook/Multi-Factor L:S/Processed_Data/subindustries.parquet').iloc[:, 0]

# normalize codes to match panel columns like 000001
industry.index = industry.index.map(_norm_code)
subindustry.index = subindustry.index.map(_norm_code)

# drop duplicate codes if any, then align to stock universe (columns), not dates (index)
industry = industry[~industry.index.duplicated(keep='last')].reindex(close_df.columns)
subindustry = subindustry[~subindustry.index.duplicated(keep='last')].reindex(close_df.columns)


In [146]:
from analytics import _rowwise_corr_nan,_factor_quantiles_by_row,_winsorize_by_row,_zscore_by_row,plot_equity_curve,performance_summary,compute_alpha_metrics_wide

In [206]:
%run "Factors.ipynb"

Note: you may need to restart the kernel to use updated packages.


## 3.2 Stock Universe

In [29]:
index_comp = pd.read_csv('/Users/apple/Desktop/PitchBook/Multi-Factor L:S/csi500_comp.csv')
index_weight = pd.read_csv('/Users/apple/Desktop/PitchBook/Multi-Factor L:S/csi500_weights.csv')

In [ ]:
# ---------- main alignment ----------
def align_data_to_csi500(data: dict,
                         index_comp: pd.DataFrame,
                         index_weight: pd.DataFrame,
                         daily_calendar_source: str = "close",
                         keep_only_common_codes: bool = True,
                         renormalize_daily_weights: bool = True):
    """
    Returns:
      aligned_data: dict of daily panels masked to CSI500 membership
      comp_daily: daily 0/1 membership
      wgt_daily: daily weights aligned (optionally renormalized to sum to 1 over available members)
      codes: final universe code list
    """

    # 1) normalize index panels
    index_comp = normalize_panel_codes(index_comp, date_col="date" if "date" in index_comp.columns else None)
    index_weight = normalize_panel_codes(index_weight, date_col="date" if "date" in index_weight.columns else None)
    index_comp = (index_comp.fillna(0).astype(int))

    # 2) get daily calendar from your price panel
    base = data[daily_calendar_source]
    base = normalize_panel_codes(base)
    daily_index = base.index

    # 3) expand membership/weights monthly -> daily
    comp_daily = expand_monthly_to_daily(index_comp, daily_index).fillna(0).astype(int)
    wgt_daily = expand_monthly_to_daily(index_weight, daily_index)

    # 4) decide universe codes
    index_codes = set(comp_daily.columns)

    if keep_only_common_codes:
        common = index_codes.copy()
        for k, df in data.items():
            if df is None:
                continue
            if isinstance(df, pd.DataFrame):
                df2 = normalize_panel_codes(df) if k not in {"industries", "subindustries"} else df
                common &= set(df2.columns) if isinstance(df2.index, pd.DatetimeIndex) else common
        codes = sorted(common)
    else:
        codes = sorted(index_codes)

    # 5) align + mask each daily panel
    aligned = {}
    for k, df in data.items():
        if df is None:
            continue

        # industries/subindustries are usually code->label tables (not time series)
        if k in {"industries", "subindustries"}:
            # support either: columns=codes (single row) or index=codes
            tmp = df.copy()
            if isinstance(tmp, pd.DataFrame):
                if set(tmp.columns) & set(codes):
                    aligned[k] = tmp.loc[:, [c for c in tmp.columns if c in codes]]
                elif set(tmp.index) & set(codes):
                    aligned[k] = tmp.loc[[c for c in tmp.index if c in codes]]
                else:
                    aligned[k] = tmp
            else:
                aligned[k] = tmp
            continue

        df = normalize_panel_codes(df)
        aligned[k] = align_and_mask_daily_panel(df, daily_index, codes, comp_daily)

    # 6) align weights to the final codes + optionally renormalize
    comp_daily = comp_daily.reindex(index=daily_index, columns=codes).fillna(0).astype(int)
    wgt_daily = wgt_daily.reindex(index=daily_index, columns=codes)

    if renormalize_daily_weights:
        # set non-members to 0 before normalization
        w = wgt_daily.where(comp_daily == 1, 0.0)
        s = w.sum(axis=1).replace(0, np.nan)
        wgt_daily = w.div(s, axis=0)  # sums to 1 across available members
    else:
        wgt_daily = wgt_daily.where(comp_daily == 1)

    return aligned, comp_daily, wgt_daily, codes


# Execute
aligned_data, csi500_comp_daily, csi500_wgt_daily, csi500_codes = align_data_to_csi500(
    data=data,
    index_comp=index_comp,
    index_weight=index_weight,
    daily_calendar_source="close",
    keep_only_common_codes=True,
    renormalize_daily_weights=True,
)

print("Final universe codes:", len(csi500_codes))
print("Daily comp panel:", csi500_comp_daily.shape, "Daily weight panel:", csi500_wgt_daily.shape)
print("Aligned panels:", {k: v.shape for k, v in aligned_data.items() if isinstance(v, pd.DataFrame) and isinstance(v.index, pd.DatetimeIndex)})

Final universe codes: 1771
Daily comp panel: (8592, 1771) Daily weight panel: (8592, 1771)
Aligned panels: {'cap': (8592, 1771), 'close': (8592, 1771), 'debt': (8592, 1771), 'high': (8592, 1771), 'low': (8592, 1771), 'net_profit': (8592, 1771), 'open': (8592, 1771), 'operating_income': (8592, 1771), 'pb': (8592, 1771), 'pe': (8592, 1771), 'returns': (8592, 1771), 'shares_outstanding': (8592, 1771), 'volume': (8592, 1771), 'vwap': (8592, 1771)}


In [30]:
cap_df = aligned_data['cap'] 
close_df = aligned_data["close"] 
open_df = aligned_data['open'] 
high_df = aligned_data['high'] 
low_df = aligned_data['low'] 
volume_df = aligned_data['volume'] 
vwap_df = aligned_data['vwap'] 
returns_df = aligned_data["returns"] 
sharesout_df = aligned_data['shares_outstanding'] 
debt_df = aligned_data['debt'] 
operating_income_df = aligned_data['operating_income']

In [31]:
pe_df = aligned_data['pe'] 
pb_df = aligned_data['pb']

In [32]:
net_profit_df = aligned_data['net_profit']

## 3.3 Factor Implementation

In [33]:
fwd_ret = close_df.shift(-1) / close_df - 1.0

## 0. MOM1

In [80]:
fac_mom_1 = mom_1(close_df)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_1486/647476319.py:28: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = close.pct_change()


In [81]:
res_mom_1 = compute_alpha_metrics_wide(
    alpha_df= fac_mom_1,  
    close_df=close_df,  
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 1. MR5

In [79]:
fac_mr_5 = mr_5(open_df,volume_df)

In [82]:
res_mr_5 = compute_alpha_metrics_wide(
    alpha_df= fac_mr_5,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 2. MR9

In [83]:
fac_mr_9 = mr_9(returns_df,cap_df,volume_df)

/opt/anaconda3/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/anaconda3/lib/python3.12/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [84]:
res_mr_9 = compute_alpha_metrics_wide(
    alpha_df= fac_mr_9,       
    close_df=close_df,  
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 3. MR3

In [85]:
fac_mr_3 = mr_3(close_df,volume_df)

In [86]:
res_mr_3 = compute_alpha_metrics_wide(
    alpha_df= fac_mr_3,         
    close_df=close_df,  
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 4. MR4

In [87]:
fac_mr_4 = mr_4(close_df,open_df,volume_df)

In [88]:
res_mr_4 = compute_alpha_metrics_wide(
    alpha_df= fac_mr_4,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 5. MR10

In [89]:
fac_mr_10 = mr_10(close_df,open_df,volume_df,sharesout_df,industry,horro_window=10)

In [90]:
res_mr_10 = compute_alpha_metrics_wide(
    alpha_df= fac_mr_10,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 6. ST3

In [91]:
fac_st_3 = st_3(volume_df,sharesout_df,cap_df)

In [92]:
res_st_3 = compute_alpha_metrics_wide(
    alpha_df= fac_st_3,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 7. MR2

In [93]:
fac_mr_2 = mr_2(close_df,high_df,low_df)

In [94]:
res_mr_2 = compute_alpha_metrics_wide(
    alpha_df= fac_mr_2,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 8. VA4

In [95]:
fac_va_4 = va_4(operating_income_df,vwap_df)

In [96]:
res_va_4 = compute_alpha_metrics_wide(
    alpha_df= fac_va_4,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 9. MR1

In [97]:
fac_mr_1 = mr_1(close_df,high_df,low_df)

In [98]:
res_mr_1 = compute_alpha_metrics_wide(
    alpha_df= fac_mr_1,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 10. VA1

In [99]:
fac_va_1 = va_1(operating_income_df,cap_df)

In [100]:
res_va_1 = compute_alpha_metrics_wide(
    alpha_df= fac_va_1,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 11. ST2

In [101]:
fac_st_2 = st_2(volume_df,sharesout_df,cap_df)

In [102]:
res_st_2 = compute_alpha_metrics_wide(
    alpha_df= fac_st_2,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 12. MR6

In [104]:
fac_mr_6 = mr_6(returns_df,subindustry,volume_df)

In [105]:
res_mr_6 = compute_alpha_metrics_wide(
    alpha_df= fac_mr_6,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

In [106]:
res_mr_6['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,4443,0.008880,0.068768,2.049943,8.607552,0.071589,0.175459,-0.100769,0.006456,0.126892
5,4439,0.008995,0.069789,2.046166,8.587826,0.113295,0.034743,-0.102744,0.007274,0.131839
10,4434,0.010291,0.069096,2.364269,9.917321,0.127638,0.085041,-0.100640,0.008220,0.131756
20,4424,0.010127,0.068315,2.353128,9.859452,0.186749,0.169173,-0.098911,0.006676,0.134401


## 13. MR7

In [107]:
fac_mr_7 = mr_7(close_df,open_df,volume_df,returns_df)

In [108]:
res_mr_7 = compute_alpha_metrics_wide(
    alpha_df= fac_mr_7,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 14. ST1

In [110]:
fac_st_1 = st_1(volume_df,sharesout_df)

In [111]:
res_st_1 = compute_alpha_metrics_wide(
    alpha_df= fac_st_1,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 15. MR8

In [113]:
fac_mr_8 = mr_8(vwap_df,close_df,volume_df)

In [114]:
res_mr_8 = compute_alpha_metrics_wide(
    alpha_df= fac_mr_8,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 16. VA2

In [116]:
fac_va_2 = va_2(returns_df,operating_income_df,cap_df)

In [117]:
res_va_2 = compute_alpha_metrics_wide(
    alpha_df= fac_va_2,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

In [118]:
res_va_2['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,3225,0.011781,0.132998,1.406181,5.030439,0.068987,0.939419,-0.207915,0.006534,0.244989
5,3221,0.015993,0.131062,1.937120,6.925509,0.027908,0.650538,-0.198527,0.014303,0.236355
10,3216,0.014524,0.131947,1.747362,6.242243,0.152673,0.994372,-0.197624,0.011537,0.236542
20,3206,0.012561,0.133050,1.498654,5.345431,0.285535,0.832553,-0.196398,0.007076,0.241403


## 17. QU1

In [119]:
fac_qu_1 = qu_1(operating_income_df,vwap_df,volume_df,returns_df)

In [120]:
res_qu_1 = compute_alpha_metrics_wide(
    alpha_df= fac_qu_1,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 18. VA3

In [122]:
fac_va_3 = va_3(operating_income_df,vwap_df)

In [123]:
res_va_3 = compute_alpha_metrics_wide(
    alpha_df= fac_va_3,         # Date x Code
    close_df=close_df,  # Date x Code (already masked to CSI500)
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,                  # CSI500 universe → use something like 200+
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 19. VA7

In [138]:
fac_va_7 = va_7(pe_df,net_profit_df,industry)

In [139]:
res_va_7 = compute_alpha_metrics_wide(
    alpha_df= fac_va_7,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

In [140]:
res_va_7['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,4401,0.005639,0.068688,1.303236,5.446260,0.044601,0.322632,-0.108222,0.004920,0.117952
5,4397,0.009244,0.070278,2.088012,8.721898,0.110806,0.392298,-0.103312,0.008826,0.127177
10,4392,0.012877,0.072141,2.833538,11.829324,0.129860,0.266133,-0.104700,0.013725,0.129888
20,4382,0.013413,0.074895,2.843054,11.855531,0.327788,0.617970,-0.102425,0.012718,0.137025


## 20. GR1

In [154]:
fac_gr_1 = gr_1(net_profit_df,subindustry)

In [155]:
res_gr_1 = compute_alpha_metrics_wide(
    alpha_df= fac_gr_1,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

In [156]:
res_gr_1['ic_summary']

,n,mean,std,ICIR,tstat,skew,kurt,p05,p50,p95
period,,,,,,,,,,
1,3897,0.000691,0.051929,0.211144,0.830317,-0.035444,-0.340525,-0.085109,0.000285,0.085882
5,3893,0.001359,0.051323,0.420354,1.652179,-0.128653,-0.218544,-0.084121,0.001876,0.083315
10,3888,0.002634,0.052146,0.801981,3.150119,-0.140544,-0.248103,-0.086907,0.004888,0.084893
20,3878,0.001716,0.051701,0.526856,2.066786,-0.068924,-0.311076,-0.082663,0.003030,0.085586


## 21. GR2

In [ ]:
fac_gr_2 = gr_2(net_profit_df,subindustry)

In [ ]:
res_gr_2 = compute_alpha_metrics_wide(
    alpha_df= fac_gr_3,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 22. VO3

In [167]:
fac_vo_3 = vo_3(returns_df)

In [169]:
res_vo_3 = compute_alpha_metrics_wide(
    alpha_df= fac_vo_3,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 23. MOM2

In [180]:
fac_mom_2 = mom_2(returns_df,cap_df,industry)

In [181]:
res_mom_2 = compute_alpha_metrics_wide(
    alpha_df= fac_mom_2,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

# 24. ST4

In [183]:
fac_st_4 = st_4(sharesout_df,volume_df,vwap_df,low_df,open_df,high_df)

In [184]:
res_st_4 = compute_alpha_metrics_wide(
    alpha_df= fac_st_4,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 25. MOM3

In [187]:
fac_mom_3 = mom_3(returns_df)

In [188]:
res_mom_3 = compute_alpha_metrics_wide(
    alpha_df= fac_mom_3,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 26. MOM4

In [207]:
fac_mom_4 = mom_4(close_df)

/opt/anaconda3/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/anaconda3/lib/python3.12/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [208]:
res_mom_4 = compute_alpha_metrics_wide(
    alpha_df= fac_mom_4,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 

## 27. MOM5

In [209]:
fac_mom_5 = mom_5(returns_df,industry)

In [210]:
res_mom_5 = compute_alpha_metrics_wide(
    alpha_df= fac_mom_5,         
    close_df=close_df, 
    periods=(1, 5, 10, 20),
    quantiles=5,
    ic_method="spearman",
    min_assets=200,          
    winsorize_alpha=(0.01, 0.99),
    zscore_alpha=False,
)

/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/Users/apple/Documents/GitHub/Systematic-Equities/analytics.py:102: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to 